## Desenvolvimento de dimensoes e fatos na modelagem snowflake

- dim_category
- dim_subcategory
- dim_brand
- dim_product
- dim_state
- dim_city
- dim_store
- dim_customer
- dim_date
- dim_payment
- fact_order_items

In [0]:
%sql
USE CATALOG proj_dw;
USE SCHEMA dw_snowflake;

In [0]:
%sql
CREATE OR REPLACE TABLE dim_category (
    category_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    product_category STRING
);

INSERT INTO dim_category (product_category)
SELECT DISTINCT product_category
FROM bronze.sales_orders

In [0]:
%sql
SELECT *
FROM dim_category

In [0]:
%sql
CREATE OR REPLACE TABLE dim_subcategory (
    subcategory_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    product_subcategory STRING,
    category_sk BIGINT
);

INSERT INTO dim_subcategory (product_subcategory, category_sk)
SELECT 
    DISTINCT b.product_subcategory, 
    c.category_sk

FROM bronze.sales_orders b

JOIN dim_category c ON b.product_category = c.product_category

In [0]:
%sql
SELECT *
FROM dim_subcategory

In [0]:
%sql
CREATE OR REPLACE TABLE dim_brand (
    brand_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    brand STRING
);

INSERT INTO dim_brand (brand)
SELECT DISTINCT brand
FROM bronze.sales_orders;

In [0]:
%sql
SELECT *
FROM dim_brand

In [0]:
%sql
CREATE OR REPLACE TABLE dim_product (
  product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  product_id STRING,
  product_name STRING,
  subcategory_sk BIGINT,
  brand_sk BIGINT
);

INSERT INTO dim_product (product_id, product_name, subcategory_sk, brand_sk)
SELECT
  DISTINCT b.product_id,
  b.product_name,
  s.subcategory_sk,
  br.brand_sk

FROM bronze.sales_orders b

JOIN dw_snowflake.dim_subcategory s ON b.product_subcategory = s.product_subcategory
JOIN dw_snowflake.dim_brand br ON b.brand = br.brand;

In [0]:
%sql
SELECT *
FROM proj_dw.dw_snowflake.dim_product

In [0]:
%sql
------- ANTES ------
-- Aqui vamos ver a tabela produtos na modelagem star
SELECT * 
FROM proj_dw.dw_star.dim_product
LIMIT 5

In [0]:
%sql
------- DEPOIS -------
-- Aqui vem a diferenca
-- Aqui vamos ver a tabela produtos na modelagem snowflake
SELECT *
FROM proj_dw.dw_snowflake.dim_product
LIMIT 5

- Quando seguimos o modelo snowflake, ele da mais flexibilidade para complementar as informacoes no futuro sem fazer com que a tamanho da tabela cresca exponencialmente. Por exemplo, imagine que queremos ter mais de uma linha da Apple, Apple Brasil e Apple US, ou seja complementar as informacoes da linha da marca, nesse caso o pais. Se fosse fazer isso dentro da modelagem star adicionariamos uma coluna na tabela dim_product, porem esse registro seria multiplicado para todos os outros registros dentro da tabela, entao o volume cresceria exponencialmente. Agora, no modelo snowflake se eu tenho uma dimensao 'marca' e eu complemento as informacoes das marcas (como pais, representante...) dentro dessa tabela (dimensao), eu só aumento o volume de linhas dentro da dimensao.

- Resposta mais formal: O modelo snowflake normaliza as dimensões, então atributos da marca (país, representante...) ficam em dim_marca e são armazenados uma única vez. No star, esses atributos ficariam em dim_product e se repetiriam para todos os produtos da marca, gerando redundância. Se uma marca passar a ter vários registros (Apple Brasil, Apple US), o star precisaria duplicar também os produtos, enquanto no snowflake só dim_marca ganha linhas. Em contrapartida, o snowflake exige mais joins, o que pode deixar as consultas mais lentas e complexas.

In [0]:
%sql
CREATE OR REPLACE TABLE dim_state (
  state_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  state_code STRING
);

INSERT INTO dim_state (state_code)
SELECT DISTINCT store_state

FROM bronze.sales_orders

WHERE store_state IS NOT NULL;

In [0]:
%sql
SELECT *
FROM dim_state

In [0]:
%sql
CREATE OR REPLACE TABLE dim_city (
  city_sk   BIGINT GENERATED ALWAYS AS IDENTITY,
  city_name STRING,
  state_sk  BIGINT
);

INSERT INTO dim_city (city_name, state_sk)
SELECT 
  DISTINCT b.store_city AS city_name,
  s.state_sk AS state_sk

FROM bronze.sales_orders b

JOIN dw_snowflake.dim_state s ON b.store_state = s.state_code

WHERE b.store_city IS NOT NULL;

In [0]:
%sql
SELECT *
FROM dim_city

In [0]:
%sql
CREATE OR REPLACE TABLE dim_store (
  store_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  store_id STRING,
  store_name STRING,
  city_sk BIGINT
);

INSERT INTO dim_store (store_id, store_name, city_sk)
SELECT 
  DISTINCT b.store_id,
  b.store_name,
  c.city_sk

FROM bronze.sales_orders b

JOIN dw_snowflake.dim_city c ON  b.store_city  = c.city_name
JOIN dw_snowflake.dim_state s ON  b.store_state = s.state_code AND c.state_sk    = s.state_sk;

In [0]:
%sql
SELECT *
FROM dim_store

In [0]:
%sql
CREATE OR REPLACE TABLE proj_dw.dw_snowflake.dim_customer (
  customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  customer_id STRING,
  customer_name STRING,
  customer_email STRING,
  customer_birthdate DATE,
  customer_gender STRING,
  customer_city STRING,
  customer_state STRING
);

INSERT INTO dim_customer (
  customer_id, 
  customer_name, 
  customer_email,
  customer_birthdate, 
  customer_gender,
  customer_city, 
  customer_state
)

SELECT DISTINCT
  customer_id, 
  customer_name, 
  customer_email,
  customer_birthdate, 
  customer_gender,
  customer_city, 
  customer_state

FROM bronze.sales_orders;

In [0]:
%sql
CREATE OR REPLACE TABLE proj_dw.dw_snowflake.dim_date (
  date_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  date_value DATE,
  year INT,
  month INT,
  day INT,
  month_name STRING
);

INSERT INTO dim_date (date_value, year, month, day, month_name)
SELECT DISTINCT
  order_date AS date_value,
  YEAR(order_date),
  MONTH(order_date),
  DAY(order_date),
  DATE_FORMAT(order_date, 'MMMM')

FROM bronze.sales_orders

WHERE order_date IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE proj_dw.dw_snowflake.dim_payment (
  payment_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  payment_method STRING,
  payment_installments INT,
  payment_status STRING
);

INSERT INTO dim_payment (payment_method, payment_installments, payment_status)

SELECT DISTINCT
  payment_method,
  payment_installments,
  payment_status

FROM proj_dw.bronze.sales_orders;

In [0]:
%sql
CREATE OR REPLACE TABLE proj_dw.dw_snowflake.fact_order_items AS
SELECT
  b.order_id,
  b.order_item_id,

  -- chaves
  d_date.date_sk        AS order_date_sk,
  d_cust.customer_sk    AS customer_sk,
  d_prod.product_sk     AS product_sk,
  d_store.store_sk      AS store_sk,
  d_pay.payment_sk      AS payment_sk,

  -- métricas
  b.quantity,
  b.unit_price,
  b.discount_value,
  b.total_gross,
  b.total_net

FROM proj_dw.bronze.sales_orders b

JOIN proj_dw.dw_snowflake.dim_date d_date
ON b.order_date   = d_date.date_value

JOIN proj_dw.dw_snowflake.dim_customer d_cust
ON b.customer_id  = d_cust.customer_id

JOIN proj_dw.dw_snowflake.dim_store d_store
ON b.store_id     = d_store.store_id

JOIN proj_dw.dw_snowflake.dim_payment d_pay
ON  b.payment_method       = d_pay.payment_method
AND b.payment_installments = d_pay.payment_installments
AND b.payment_status       = d_pay.payment_status

JOIN proj_dw.dw_snowflake.dim_product d_prod
ON b.product_id = d_prod.product_id;


In [0]:

%sql
SELECT *
FROM proj_dw.dw_snowflake.fact_order_items
LIMIT 5

In [0]:
%sql
SELECT *
FROM proj_dw.dw_star.fact_order_items
LIMIT 5

- Perceba que as tabelas fatos nao mudaram nada quando se compara a tabela fato feita no modelo star VS modelo snowflake
- A principal vantagem é o MODELO SEMANTICO: A maneira com que é feita a analise de dados (ferramenta de BI) se organiza SERÁ a mais eficiente, faz as consultas de forma mais eficientes, por mais que a quantidade de joins seja maior.
- O nível de complexidade do Snowflake Schema é bem maior pelo fato de normalizar as dimensoes.

Resposta mais formal:
- A tabela fato não muda entre o modelo star e o snowflake; a diferença está nas dimensões.
- Principal vantagem do snowflake: reduz redundância e facilita a manutenção das dimensões, pois cada atributo é armazenado uma única vez.
- Desvantagem: a normalização aumenta a complexidade do modelo e o número de joins, o que pode deixar as consultas mais lentas.